# Task 1: reconocer alumnos desde fotografías

Primera versión local: **foto → rostro → descriptor → comparación → nombre o NO_CONCLUYENTE**.
Usamos YuNet y SFace preentrenados en CPU. Las fotos del curso forman una galería; no entrenamos una red nueva.
Este notebook no utiliza ROS, Gazebo ni controla el robot. Task 2 queda intacto.

Ejecuta las celdas en orden, seleccionando la `.venv` compartida de la raíz de Proyecto2.
Las fotografías originales se conservan. Los datos derivados van a `datos/` y `resultados/`, excluidos de Git.
Las fotos no se muestran por defecto, para evitar persistirlas en las salidas.

**1. Verificar entorno y rutas.** Puedes abrir el notebook con directorio de trabajo en la raíz de Proyecto2 o en `task 1`.
Los módulos auxiliares contienen la lógica reutilizable; aquí veremos cada etapa y sus resultados.
No instales otro paquete OpenCV: el entorno ya utiliza `opencv-contrib-python`.

In [ ]:
import sys
from pathlib import Path

actual = Path.cwd().resolve()
TAREA_DIR = next((p for p in (actual, actual / 'task 1')
                  if (p / 'reconocimiento_facial.py').is_file() and (p / 'rostros').is_dir()), None)
if TAREA_DIR is None:
    raise RuntimeError('Abre el notebook desde Proyecto2 o desde su carpeta task 1.')
PROYECTO_DIR = TAREA_DIR.parent
if Path(sys.prefix).resolve() != (PROYECTO_DIR / '.venv').resolve():
    raise RuntimeError('Selecciona el kernel de la .venv global de Proyecto2 y reinicia.')
if str(TAREA_DIR) not in sys.path:
    sys.path.insert(0, str(TAREA_DIR))

from collections import Counter
from dataclasses import asdict
import csv
import json
from html import escape
import time
import cv2
import numpy as np
from IPython.display import HTML, Image, display
from modelos_rostros import MODELOS, preparar_modelos, sha256_archivo
from reconocimiento_facial import (inventariar, preparar_particion, ExtractorFacial,
                                  Galeria, casos_de_rol, calibrar, evaluar, decidir)

cv2.setNumThreads(2)
DATOS = TAREA_DIR / 'datos'
RESULTADOS = TAREA_DIR / 'resultados'
DATOS.mkdir(exist_ok=True)
RESULTADOS.mkdir(exist_ok=True)

def tabla(filas, columnas=None):
    if not filas:
        print('Sin filas que mostrar.')
        return
    columnas = columnas or list(filas[0])
    def texto(valor):
        if valor is None:
            return '-'
        if isinstance(valor, (float, np.floating)):
            return f'{valor:.4f}'
        return str(valor)
    cabecera = ''.join(f'<th>{escape(str(c))}</th>' for c in columnas)
    cuerpo = ''.join('<tr>' + ''.join(f'<td>{escape(texto(f.get(c)))}</td>' for c in columnas) + '</tr>'
                     for f in filas)
    display(HTML('<table border="1" cellpadding="5"><thead><tr>' + cabecera +
                 '</tr></thead><tbody>' + cuerpo + '</tbody></table>'))

print('Python:', sys.version.split()[0], '| OpenCV:', cv2.__version__)
print('Entorno compartido:', sys.prefix)
print('Carpeta de trabajo:', TAREA_DIR)

**2. Preparar los dos modelos oficiales.** La primera ejecución descarga unos 39 MB si faltan.
Después funciona sin internet. Cada archivo se verifica con su tamaño y SHA-256 publicado en OpenCV Zoo.
Un archivo existente incorrecto produce un error y no se reemplaza silenciosamente.

Usamos YuNet **2023mar**, compatible con OpenCV 4.x, y SFace **2021dec**.
Fuentes: [YuNet](https://github.com/opencv/opencv_zoo/tree/main/models/face_detection_yunet),
[SFace](https://github.com/opencv/opencv_zoo/tree/main/models/face_recognition_sface) y
[flujo de OpenCV](https://docs.opencv.org/4.x/d0/dd4/tutorial_dnn_face.html).

In [ ]:
PERMITIR_DESCARGA = True  # No hace conexiones si los dos modelos ya están preparados.
rutas_modelos = preparar_modelos(TAREA_DIR / 'modelos', permitir_descarga=PERMITIR_DESCARGA)
tabla([{'modelo': nombre, 'archivo': ruta.name, 'bytes': ruta.stat().st_size, 'SHA256': 'verificado'}
       for nombre, ruta in rutas_modelos.items()])

**3. Inventariar las fotografías.** Cada subcarpeta de `rostros/` es la etiqueta de un alumno.
Comprobamos duplicados exactos antes de repartir; no borramos ni movemos originales.
Este inventario todavía no demuestra que una foto contenga una cara reconocible.

In [ ]:
CARPETA_ROSTROS = TAREA_DIR / 'rostros'
fotos = inventariar(CARPETA_ROSTROS)
cuenta = Counter(f.alumno for f in fotos)
tabla([{'alumno': alumno, 'fotos': cantidad} for alumno, cantidad in sorted(cuenta.items())])
print(f'Total: {len(fotos)} fotos de {len(cuenta)} alumnos.')

**4. Separar galería, calibración y evaluación antes de reconocer.**
La división inicial es determinista por nombre de archivo: las últimas dos imágenes de cada alumno
se reservan para calibración y evaluación, y las demás forman la galería.
Con cinco fotos son 3/1/1; con seis, 4/1/1. Con menos fotos puede faltar alguna etapa.

**Es un ensayo exploratorio por fotografía, no una separación por sesiones independientes.**
Fotos tomadas con pocos segundos de diferencia pueden dar resultados optimistas.
Revisa `datos/particion_rostros.json` y posteriormente añade consultas de otra toma/impresión.
La misma foto nunca debe estar en dos conjuntos.

El manifiesto se conserva entre ejecuciones. Si añades o cambias fotos, revisa el reparto,
cambia `REGENERAR_PARTICION` a `True` y vuelve a ejecutar desde esta etapa; luego déjalo en `False`.
Recalibra y reevalúa todo. No cambies conjuntos o parámetros para arreglar un error de evaluación.

In [ ]:
REGENERAR_PARTICION = False
ARCHIVO_PARTICION = DATOS / 'particion_rostros.json'
particion = preparar_particion(fotos, ARCHIVO_PARTICION, regenerar=REGENERAR_PARTICION)
conteos = Counter((f.alumno, f.rol) for f in particion)
tabla([{'alumno': alumno, **{rol: conteos[(alumno, rol)]
                           for rol in ('galeria', 'calibracion', 'evaluacion')}}
       for alumno in sorted(cuenta)])
tabla([{'archivo': f.ruta, 'rol': f.rol} for f in particion])
print('Manifiesto:', ARCHIVO_PARTICION)

**5. Detectar, alinear y describir.** Procesamos una imagen a la vez, reducida en memoria a un máximo de 960 píxeles por lado.
YuNet debe encontrar exactamente una cara suficientemente grande; SFace extrae un vector del rostro alineado.
Cero caras, varias caras y errores de lectura/extracción se registran como rechazos, sin elegir otra foto a escondidas.

`deteccion` mide la confianza del detector, no la probabilidad de que sea un alumno.
`nitidez` es un diagnóstico de la imagen alineada, no un umbral de identidad.
`tiempo_ms` incluye reescalado y reconocimiento, pero no lectura/decodificación del archivo ni carga inicial de modelos.

In [ ]:
CONFIG_EXTRACTOR = {'lado_maximo': 960, 'deteccion_minima': 0.85, 'cara_minima': 32}
extractor = ExtractorFacial(rutas_modelos, **CONFIG_EXTRACTOR)
inicio = time.perf_counter()
analisis = [extractor.analizar_archivo(CARPETA_ROSTROS / f.ruta, f) for f in particion]
auditoria = [a.fila() for a in analisis]
tabla(auditoria)
print('Estados:', dict(Counter(a.estado for a in analisis)))
print(f'Tiempo total con lectura de archivos: {time.perf_counter() - inicio:.2f} s')
with (DATOS / 'auditoria.csv').open('w', newline='', encoding='utf-8') as archivo:
    escritor = csv.DictWriter(archivo, fieldnames=list(auditoria[0]))
    escritor.writeheader()
    escritor.writerows(auditoria)

**6. Crear la galería solo con referencias.** Cada vector se normaliza.
La puntuación de un alumno será la media de sus dos referencias más parecidas a la consulta
(o de una si solo tiene una referencia). Es una regla fija que mantendremos al calibrar y evaluar.
Si un alumno no tiene ninguna referencia válida, hay que revisar sus fotos antes de seguir.

La galería numérica se guarda en `datos/galeria.npz`, fuera de Git; no contiene las fotografías.
No es un nuevo entrenamiento de SFace.

In [ ]:
galeria = Galeria.desde_analisis(analisis)
reservadas = {f.ruta for f in particion if f.rol != 'galeria'}
assert set(galeria.origenes).isdisjoint(reservadas)
galeria.guardar(DATOS / 'galeria.npz')
tabla([{'alumno': alumno, 'referencias_validas': int(np.sum(galeria.alumnos == alumno))}
       for alumno in sorted(set(galeria.alumnos.tolist()))])
print('Forma de los descriptores:', galeria.vectores.shape)

**7. Calibrar dos condiciones, sin mirar las fotos de evaluación.**
Aceptaremos el mejor candidato solo si su similitud supera un umbral y se separa suficientemente del segundo.
Los empates se rechazan. Las puntuaciones no son porcentajes ni probabilidades.

Usamos únicamente las consultas de calibración. Para ensayar rechazo, quitamos temporalmente la identidad
real de la galería al puntuar esa misma consulta. Esto simula una identidad no inscrita; **no reemplaza probar
personas nuevas y otras sesiones**. Se necesitan al menos tres identidades con referencias y consultas útiles.

El ajuste prioriza evitar nombres erróneos/aceptaciones de esos negativos y después maximiza aciertos.
Es conservador y puede rechazar una foto válida. Si rechaza todas las consultas, hacen falta mejores datos
antes de dar por terminado el reconocimiento.

In [ ]:
casos_calibracion = casos_de_rol(analisis, galeria, 'calibracion')
criterio, resumen_calibracion = calibrar(casos_calibracion)
tabla([{'umbral_coseno': criterio.umbral, 'margen_minimo': criterio.margen_minimo}])
tabla([resumen_calibracion])
configuracion = {
    'version': 1, 'criterio': asdict(criterio), 'extractor': CONFIG_EXTRACTOR,
    'regla_puntuacion': 'coseno, media de hasta 2 mejores referencias por alumno',
    'modelos_sha256': {k: v['sha256'] for k, v in MODELOS.items()},
    'particion_sha256': sha256_archivo(ARCHIVO_PARTICION),
    'calibracion': resumen_calibracion,
}
(DATOS / 'criterio.json').write_text(json.dumps(configuracion, indent=2), encoding='utf-8')
if resumen_calibracion['aciertos'] == 0:
    print('Atencion: el criterio solo rechaza en esta calibracion. Revisar datos antes de usarlo.')

**8. Evaluar con las fotos reservadas.** El criterio anterior queda fijo.
Contamos tanto nombres correctos como errores y rechazos; un fallo del detector también cuenta.
Se informa aparte del rechazo simulado retirando la identidad real. No confundas este ensayo
con un conjunto real de personas desconocidas.

Una consulta por alumno es una prueba pequeña dentro de esta colección. No demuestra precisión general,
y todavía faltan fotografías impresas vistas por cámara y datos de otros días.

In [ ]:
casos_evaluacion = casos_de_rol(analisis, galeria, 'evaluacion')
filas_evaluacion, resumen_evaluacion = evaluar(casos_evaluacion, criterio)
tabla(filas_evaluacion, ['archivo', 'real', 'nombre', 'candidato', 'similitud', 'margen', 'motivo', 'correcto'])
tabla([resumen_evaluacion])
reporte = {'configuracion': configuracion, 'resumen': resumen_evaluacion, 'filas': filas_evaluacion,
           'limite': 'ensayo pequeno por foto; no validacion independiente por sesion ni personas nuevas'}
(RESULTADOS / 'evaluacion.json').write_text(json.dumps(reporte, ensure_ascii=False, indent=2), encoding='utf-8')
print(f"Resultado: {resumen_evaluacion['aciertos']}/{resumen_evaluacion['consultas']} nombres correctos; "
      f"{resumen_evaluacion['rechazos']} rechazos; {resumen_evaluacion['nombres_erroneos']} nombres erroneos.")

**9. Consultar una fotografía.** Por defecto usamos la primera reservada para evaluación.
Puedes cambiar `CONSULTA` por una ruta a otra imagen. Si usas una referencia de galería,
el resultado se señala como demostración y no sirve para medir precisión.

`MOSTRAR_FOTO` está desactivado para no guardar imágenes personales en las salidas.
Si lo activas, podrás ver la cara localizada; limpia esa salida antes de guardar/compartir el notebook.
Un `candidato` mostrado en una respuesta NO_CONCLUYENTE no es una identidad aceptada.

In [ ]:
consultas_reservadas = [f for f in particion if f.rol == 'evaluacion']
if not consultas_reservadas:
    raise RuntimeError('No hay consultas reservadas. Hace falta otra foto por alumno.')
CONSULTA = CARPETA_ROSTROS / consultas_reservadas[0].ruta
# Alternativa: CONSULTA = TAREA_DIR / 'datos' / 'otra_foto.jpg'
MOSTRAR_FOTO = False

if not CONSULTA.is_file():
    raise FileNotFoundError(CONSULTA)
hash_consulta = sha256_archivo(CONSULTA)
if hash_consulta in {f.sha256 for f in particion if f.rol == 'galeria'}:
    print('Aviso: la consulta es una referencia de galeria; esto no mide generalizacion.')
consulta = extractor.analizar_archivo(CONSULTA)
ranking = galeria.puntuar(consulta.descriptor) if consulta.estado == 'OK' else []
respuesta = decidir(ranking, criterio)
if consulta.estado != 'OK':
    respuesta['motivo'] = consulta.estado
tabla([respuesta])
tabla([{'alumno': nombre, 'similitud': valor} for nombre, valor in ranking])

if MOSTRAR_FOTO:
    imagen = cv2.imdecode(np.fromfile(CONSULTA, dtype=np.uint8), cv2.IMREAD_COLOR)
    if imagen is not None:
        if consulta.caja is not None:
            x, y, w, h = map(round, consulta.caja)
            cv2.rectangle(imagen, (x, y), (x + w, y + h), (0, 200, 0), 4)
        escala = min(1.0, 640 / max(imagen.shape[:2]))
        imagen = cv2.resize(imagen, None, fx=escala, fy=escala)
        ok, jpeg = cv2.imencode('.jpg', imagen)
        if ok:
            display(Image(data=jpeg.tobytes()))

**10. Comprobar una entrada sin rostro.** Una imagen negra debe producir un rechazo,
no el nombre del alumno más parecido. Esto comprueba un caso de fallo, no la precisión con personas nuevas.

In [ ]:
prueba_vacia = extractor.analizar_imagen(np.zeros((480, 640, 3), dtype=np.uint8))
assert prueba_vacia.estado == 'SIN_ROSTRO'
assert prueba_vacia.descriptor is None
print('Imagen sin rostro:', prueba_vacia.estado, '→ no se asigna identidad.')

**Qué hemos obtenido y qué falta.** Tenemos una galería, un criterio calibrado, una evaluación reservada
y una consulta configurable. Los originales siguen en `rostros/`; todos los derivados están en `datos/` y `resultados/`.

El siguiente ensayo será una **foto impresa distinta de las referencias vista por webcam**, variando distancia,
reflejos e inclinación. Después se podrá usar la cámara del robot. La navegación autónoma, el retorno a A
y el anuncio del nombre son etapas independientes y no se implementan en este notebook.

Pruebas del módulo, desde la raíz de Proyecto2:

```bash
.venv/bin/python -m unittest discover -s "task 1/tests" -t "task 1" -v
```

Antes de compartir, limpia las salidas con imágenes si activaste `MOSTRAR_FOTO`.
Cuando incorpores alumnos, revisa la partición y repite galería, calibración y evaluación.

**11. Prueba opcional con una fotografía externa.**

Prueba incorporada a partir del trabajo de Landov311 en
[`paz_y_rostros`](https://github.com/Manueljsilva/turtlebot4-vision-proyecto/pull/1).
Ejecuta primero las secciones anteriores para cargar modelos, galería y criterio.

Coloca la imagen en `task 1/datos/rostro_prueba_1.jpeg`, o cambia `CONSULTA_EXTERNA`.
Si el archivo no existe, esta prueba se omite y el notebook puede terminar normalmente.
La consulta muestra el nombre aceptado o NO_CONCLUYENTE, el motivo y el ranking completo.
Reutiliza el criterio calibrado: esta foto no se añade a la galería ni modifica los umbrales.
Una foto usada como referencia no sirve para medir generalización.

`MOSTRAR_FOTO_EXTERNA=False` evita guardar una imagen personal en las salidas.
Puedes activarlo localmente para ver la cara localizada; al compartir, limpia las salidas
de la copia que vayas a subir. Las fotografías originales se conservan fuera de Git.


In [ ]:
# Configuración de la consulta; no cambia la galería ni el criterio.
necesarios = ('TAREA_DIR', 'extractor', 'galeria', 'criterio', 'particion',
              'sha256_archivo', 'decidir', 'tabla', 'cv2', 'np', 'display', 'Image')
if any(nombre not in globals() for nombre in necesarios):
    raise RuntimeError('Ejecuta primero las secciones 1 a 7 del notebook.')

CONSULTA_EXTERNA = TAREA_DIR / 'datos' / 'rostro_prueba_1.jpeg'
MOSTRAR_FOTO_EXTERNA = False

if not CONSULTA_EXTERNA.is_file():
    print('Prueba externa opcional sin ejecutar: falta la imagen.')
    print('Coloca tu foto en:', CONSULTA_EXTERNA)
    print('También puedes cambiar CONSULTA_EXTERNA en esta celda.')
else:
    hash_externo = sha256_archivo(CONSULTA_EXTERNA)
    if hash_externo in {f.sha256 for f in particion if f.rol == 'galeria'}:
        print('Aviso: esta foto es una referencia de galería; no mide generalización.')

    consulta_externa = extractor.analizar_archivo(CONSULTA_EXTERNA)
    ranking_externo = (galeria.puntuar(consulta_externa.descriptor)
                       if consulta_externa.estado == 'OK' else [])
    respuesta_externa = decidir(ranking_externo, criterio)
    if consulta_externa.estado != 'OK':
        respuesta_externa['motivo'] = consulta_externa.estado

    print('Decisión de identidad:')
    tabla([respuesta_externa])
    if ranking_externo:
        print('Ranking completo de similitud; las puntuaciones no son probabilidades:')
        tabla([{'alumno': nombre, 'similitud': valor}
               for nombre, valor in ranking_externo])

    if MOSTRAR_FOTO_EXTERNA and consulta_externa.estado == 'OK':
        imagen_externa = cv2.imdecode(np.fromfile(CONSULTA_EXTERNA, dtype=np.uint8),
                                     cv2.IMREAD_COLOR)
        if imagen_externa is not None:
            if consulta_externa.caja is not None:
                x, y, w, h = map(round, consulta_externa.caja)
                cv2.rectangle(imagen_externa, (x, y), (x + w, y + h), (0, 200, 0), 4)
            escala = min(1.0, 640 / max(imagen_externa.shape[:2]))
            imagen_externa = cv2.resize(imagen_externa, None, fx=escala, fy=escala)
            ok, jpeg_externo = cv2.imencode('.jpg', imagen_externa)
            if ok:
                display(Image(data=jpeg_externo.tobytes()))
